# APOLLO 17 SUBSURFACE TEMPERATURES, 1973–1974


## DATA

Gradient-thermometer measurements from the NASA Planetary Data System [Apollo 15/17 HFE Concatenated Data bundle](https://pds-geosciences.wustl.edu/missions/apollo/a15_17_hfe_concatenated.htm), version 1.0, DOI [10.17189/1518441](https://doi.org/10.17189/1518441). The exact PDS split products are [`a17p1f1_split.tab`](https://pds-geosciences.wustl.edu/lunar/urn-nasa-pds-a15_17_hfe_concatenated/data/split/a17p1f1_split.tab), [`a17p1f2_split.tab`](https://pds-geosciences.wustl.edu/lunar/urn-nasa-pds-a15_17_hfe_concatenated/data/split/a17p1f2_split.tab), [`a17p2f1_split.tab`](https://pds-geosciences.wustl.edu/lunar/urn-nasa-pds-a15_17_hfe_concatenated/data/split/a17p2f1_split.tab), and [`a17p2f2_split.tab`](https://pds-geosciences.wustl.edu/lunar/urn-nasa-pds-a15_17_hfe_concatenated/data/split/a17p2f2_split.tab).

The PDS split products preserve Earth-received UTC timestamps, replace differential temperature with the corrected value where available, remove documented missing values and most flagged artifacts, and provide the individual A/B sensor temperatures. For NSSDC records these are reconstructed from bridge mean and corrected differential temperature; later reconstructed records use explicit archived sensor temperatures. PDS names the columns `TG...`; this example uses the requested `DTG...` notation. Sensor depths come from the bundle documentation.

The repository includes [local source extracts and preprocessing notes](../data/apollo17_hfe/README.md) for every record from 1973-02-01 through 1974-12-31, the shared stable interval available for all four bridges. [`generate.py`](../data/apollo17_hfe/generate.py) masks flagged values and inserts line breaks for outages longer than six hours, then writes the compact `plot_data.npz` loaded below. It neither averages, resamples, nor interpolates.


In [1]:
from pathlib import Path
import numpy as np

repo = Path("..") if Path.cwd().name == "examples" else Path(".")
hfe = np.load(repo / "data/apollo17_hfe/plot_data.npz")

## PLOT


In [2]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import lab74

lab74.use(line_series="grayscale")
out = repo / "examples/output"
out.mkdir(exist_ok=True)

fig, axes = plt.subplots(2, 1, figsize=(6.5, 6.0), sharex=True, sharey=True)
series = {
    1: [("DTG11A", 130), ("DTG11B", 177), ("DTG12A", 185), ("DTG12B", 233)],
    2: [("DTG21A", 131), ("DTG21B", 178), ("DTG22A", 186), ("DTG22B", 234)],
}
all_times = np.concatenate(
    [hfe[f"{name}_time"] for rows in series.values() for name, _ in rows]
)
all_times = all_times[~np.isnat(all_times)]
time_limits = (
    all_times.min().astype("datetime64[Y]"),
    all_times.max().astype("datetime64[Y]") + np.timedelta64(1, "Y"),
)
label_time = np.datetime64("1973-04-15")
label_sides = {
    1: {"DTG12B": "above", "DTG12A": "above", "DTG11B": "below", "DTG11A": "above"},
    2: {"DTG22B": "above", "DTG22A": "above", "DTG21B": "below", "DTG21A": "below"},
}

for probe, ax in enumerate(axes, start=1):
    for name, depth in series[probe]:
        x = hfe[f"{name}_time"]
        y = hfe[f"{name}_temperature"]
        (line,) = ax.plot(x, y, linestyle="-", marker="")
        color = line.get_color()

        valid = np.isfinite(y) & ~np.isnat(x)
        valid_indices = np.flatnonzero(valid)
        label_index = valid_indices[np.argmin(np.abs(x[valid] - label_time))]
        side = label_sides[probe][name]
        label_distance = 3 if name == "DTG22A" else 5
        lab74.direct_label(
            ax,
            x[label_index],
            y[label_index],
            f"{depth} cm",
            offset=(0, label_distance if side == "above" else -label_distance),
            color=color,
            ha="center",
            va="bottom" if side == "above" else "top",
        )

    lab74.plate_label(
        ax,
        f"Apollo 17 Probe {probe}",
        style="emphasized",
        loc="upper left",
    )
    ax.set_xlim(*time_limits)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.xaxis.set_minor_locator(mdates.MonthLocator())
    ax.yaxis.set_major_locator(mticker.MultipleLocator(0.5))
    ax.yaxis.set_minor_locator(mticker.MultipleLocator(0.1))
    lab74.format_frame(ax, style="open")

axes[-1].set_xlabel("TIME, CALENDAR YEAR")
fig.supylabel("TEMPERATURE, K")
fig.subplots_adjust(hspace=0.08)
fig.savefig(out / "02_apollo17_hfe.png")
plt.close(fig)

/var/folders/8h/470mzdmj1sz2h00yb93qmvfc0000gn/T/ipykernel_7347/2272805842.py:14: UserWarning: no explicit representation of timezones available for np.datetime64
  records["time"].astype("datetime64[ms]") for records in bridges.values()
/var/folders/8h/470mzdmj1sz2h00yb93qmvfc0000gn/T/ipykernel_7347/3210382693.py:44: UserWarning: no explicit representation of timezones available for np.datetime64
  time = records["time"].astype("datetime64[ms]")
/var/folders/8h/470mzdmj1sz2h00yb93qmvfc0000gn/T/ipykernel_7347/3210382693.py:48: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  return np.insert(time, gap_after, np.datetime64("NaT")), np.insert(
